## 大模型后训练中的RL算法

本节按梳理大模型后训练里最常见的三条路线：`PPO`、`DPO`、`GRPO`，后续的示例和实现参考代码库`huggingface/trl`


1. `PPO`是RLHF中的经典算法，在`instruct-gpt`中就有使用.
2. `DPO`在23年的一篇论文中提出，通过直接优化偏好数据来实现。
3. `GRPO`是24年在`DeepSeek-Math`模型中提出，通过组内相对优势估计来简化训练过程。

本章节就这三个算法的原理、实现和应用场景进行详细介绍。

### 1. PPO (Proximal Policy Optimization) 

PPO是最经典的 RLHF 在线算法之一，PPO 需要 策略模型、奖励模型、参考模型和价值模型，其工程复杂度和显存成本都显著高于 DPO/GRPO。

#### 1.1 PPO算法在LLM中的适配

大语言模型的生成过程是一个由给定的`prompt`逐`token`生成`completion`的过程。则在PPO这一RL语境中。策略模型就是要训练的大语言模型本身，初始状态就是`prompt`，动作可以看作是逐`token`生成，生成结束，轨迹就是整段的`completion`。奖励值则来自`reward model`. 整个状态转移的过程图示如下

![pasted-image-1773051626990.webp](https://files.seeusercontent.com/2026/03/09/gW7z/pasted-image-1773051626990.webp)

但是在大多数RLHF设置中，并不会每生成1个`token`就给1次奖励，而是在一整个轨迹完成后，调用奖励模型对整个`completion`进行打分，中间生成token的奖励都赋予0。最终的奖励落到最后1个`token`即`<EOS>`中或者长度截断后最后1个`token`，再将最终的奖励计算的优势通过GAE向前进行传播。


从前面的章节了解到：TD误差为$\delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$,优势的递推公式为$\hat A_t = \delta_t + \gamma \lambda \hat A_{t+1}$,最后一步的优势即为奖励$\hat A_T = \delta_T$.便可递归推导出前面每一个token的优势值$\hat A_t$.

但是在一般RLHF训练中，会加上 KL 散度惩罚项来限制策略更新的幅度。则中间每个位置的reward不在为0，而是$r_t = -\beta \, \mathrm{KL}_t \quad (t<T)$,最后一步$r_T = r_{\text{rm}} - \beta \, \mathrm{KL}_T$,这样做可以使GAE的传播更为平滑.

PPO在LLM训练中涉及四个模型，policy model即我们要训练的LLM，critic model即价值模型，负责拟合$V_\phi(s_t)$,也需要在训练过程中做参数更新，reward model，负责给输出的completion打分,一般冻结。reference model负责计算KL散度，一般也是冻结。

critic model一般与policy model使用相同的主体结构，只在最后一层设置不同的输出维度，policy model输出下一个token在词表中的概率，critic model输出状态的价值估计。reward model的训练通常是在一个独立的阶段进行，使用标注数据或人类反馈来优化其打分能力。

> reference:
> 1. [PPO for LLMs: A Guide for Normal People](https://cameronrwolfe.substack.com/p/ppo-llm)
> 2. [Secrets of RLHF in Large Language Models Part I: PPO](https://arxiv.org/abs/2307.04964)

#### 1.2 TRL对PPO算法的支持

当前TRL(v0.29.0)文档里的 PPO 入口是：

```python
from trl.experimental.ppo import PPOConfig, PPOTrainer
```

在`examples/scripts/ppo/ppo.py`目录下官方提供了示例代码

当前构造器要求把主要组件显式传入：

1. `model`：policy model，负责生成回答。
2. `ref_model`：reference policy，用于计算 KL 约束；如果为 `None`，TRL 会从当前 policy 复制一个参考模型。
3. `reward_model`：奖励模型，对 `prompt + completion` 打分。
4. `value_model`：值函数模型，用于估计状态价值并做 advantage / return 相关计算。
5. `processing_class`：通常是 tokenizer 或 processor。
6. `train_dataset`：当前官方示例使用的是 **prompt-only** 数据集，训练前先预 tokenize 为 `input_ids`。


当前 TRL 中的训练数据流可分为以下几步：

1. 从数据集采样一批 `prompt`。
2. policy 基于 prompt 在线生成 `completion`。
3. reward model 对 `prompt + completion` 给出分数 `score`。
4. ref model 计算参考 log-prob，用 KL 惩罚限制策略漂移。
5. value model 估计 token 级价值，结合 `score - KL penalty` 计算 returns / advantages。
6. PPO surrogate loss 更新 policy，value loss 更新 value model。

因此 PPO 的本质仍是“在线采样 + 带 KL 正则的 actor-critic”。只是当前 TRL 把 rollout、打分、统计日志都收敛到了统一 trainer 里。


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from trl.experimental.ppo import PPOConfig, PPOTrainer


def build_ppo_dataset():
    return Dataset.from_dict(
        {
            "prompt": [
                "请总结下面这段用户反馈，并给出一句简短回复。",
                "请把下面的问题回答得更礼貌、更有帮助。",
            ]
        }
    )


def run_ppo_example():
    policy_name = "Qwen/Qwen2.5-0.5B-Instruct"
    reward_model_name = "reward-model-checkpoint"
    value_model_name = "value-model-checkpoint"

    tokenizer = AutoTokenizer.from_pretrained(policy_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    raw_dataset = build_ppo_dataset()

    def tokenize(example):
        encoded = tokenizer(example["prompt"], padding=False)
        return {
            "input_ids": encoded["input_ids"],
            "lengths": len(encoded["input_ids"]),
        }

    train_dataset = raw_dataset.map(tokenize, remove_columns=["prompt"])

    policy = AutoModelForCausalLM.from_pretrained(
        policy_name,
        torch_dtype=torch.bfloat16,
    )
    ref_policy = AutoModelForCausalLM.from_pretrained(
        policy_name,
        torch_dtype=torch.bfloat16,
    )
    reward_model = AutoModelForSequenceClassification.from_pretrained(
        reward_model_name,
        num_labels=1,
        torch_dtype=torch.bfloat16,
    )
    value_model = AutoModelForSequenceClassification.from_pretrained(
        value_model_name,
        num_labels=1,
        torch_dtype=torch.bfloat16,
    )

    training_args = PPOConfig(
        output_dir="./ppo_results",
        learning_rate=3e-6,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        total_episodes=1000,
        num_ppo_epochs=4,
        num_mini_batches=1,
        response_length=128,
        stop_token="eos",
        missing_eos_penalty=1.0,
        kl_coef=0.05,
    )

    trainer = PPOTrainer(
        args=training_args,
        processing_class=tokenizer,
        model=policy,
        ref_model=ref_policy,
        reward_model=reward_model,
        value_model=value_model,
        train_dataset=train_dataset,
    )
    return trainer


trainer = run_ppo_example()
trainer.train()

### 2. DPO (Direct Preference Optimization) 

DPO不需要在线采样，不需要显式奖励模型，也不需要 value model；只需要提供同一 prompt 下的优选回答和劣选回答。其最早在论文[Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)中提出。

#### 2.1 DPO优化目标

DPO 不训练显式 reward model，而是比较 policy 与 reference 在 `chosen/rejected` 上的相对对数概率差：

$$
L_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y^+, y^-)}\left[
\log \sigma \left(
\beta \left(
\log \frac{\pi_\theta(y^+ \mid x)}{\pi_{\mathrm{ref}}(y^+ \mid x)}
-
\log \frac{\pi_\theta(y^- \mid x)}{\pi_{\mathrm{ref}}(y^- \mid x)}
\right)
\right)
\right]
$$

其中$x$是输入的prompt，$y^+$和$y^-$分别是优选和劣选的回答。直觉上，DPO 要做的是：让模型相对于 reference，更偏向 `chosen`，更远离 `rejected`。

#### 2.2 TRL库对DPO算法的支持

TRL对DPO提供了较为稳定的实现，在`trl.trainer.dpo_trainer`模块中。

其训练使用的数据格式一般为下列形式：

```json
{
  "prompt": [{"role": "user", "content": "法国的首都是哪里？"}],
  "chosen": [{"role": "assistant", "content": "法国的首都是巴黎。"}],
  "rejected": [{"role": "assistant", "content": "法国的首都是伦敦。"}]
}
```

当前TRL的DPO接口较为简洁，最常见的最小用法是：

```python
from trl import DPOTrainer

trainer = DPOTrainer(
    model="Qwen/Qwen3-0.6B",
    train_dataset=dataset,
    processing_class=tokenizer,
)
```

当前 TRL 的 `DPOConfig` 已经不只是最初论文里的单一 sigmoid DPO，而是扩展成一个统一的 preference optimization 框架，提供了多种变体。常用增强点包括：

- `loss_type=["sigmoid"]`：标准 DPO。
- `loss_type=["sigmoid", "bco_pair", "sft"]`：支持多损失组合。
- `precompute_ref_log_probs=True`：先离线预计算 reference log-prob，减少训练时显存压力。
- `sync_ref_model=True`：支持 TR-DPO 风格的 reference 同步。
- `use_weighting=True`：支持 WPO 风格加权。

也就是说，在当前 TRL 中，`DPOTrainer` 已经成为很多偏好优化算法的统一入口，而不仅仅是“原始 DPO”。

如果训练数据是离线偏好对，且你不想引入在线采样和奖励函数，DPO 仍然是最优先的起点。它的优点是：

1. 实现简单。
2. 训练稳定。
3. 对 GPU 和工程资源要求低于 PPO/GRPO。
4. 很适合从 SFT 模型继续做偏好对齐。

在`examples/scripts/dpo.py`中，TRL为DPO的训练提供了示例实现。

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from trl import DPOConfig, DPOTrainer


def build_dpo_dataset():
    return Dataset.from_list(
        [
            {
                "prompt": "法国的首都是哪里？",
                "chosen": "法国的首都是巴黎。",
                "rejected": "法国的首都是伦敦。",
            },
            {
                "prompt": "把下面这句话改得更礼貌：把报告发我。",
                "chosen": "麻烦你把报告发我一下，谢谢。",
                "rejected": "把报告现在发给我。",
            },
        ]
    )


def run_dpo_example():
    model_name = "Qwen/Qwen3-0.6B"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    train_dataset = build_dpo_dataset()

    training_args = DPOConfig(
        output_dir="./dpo_results",
        beta=0.1,
        loss_type=["sigmoid"],
        learning_rate=1e-6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_length=1024,
        precompute_ref_log_probs=True,
    )

    trainer = DPOTrainer(
        model=model_name,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )
    return trainer


trainer = run_dpo_example()
trainer.train()

### 3. GRPO (Group Relative Policy Optimization) 

GRPO 是当前 TRL 中在线后训练最活跃的路线之一，尤其适合数学推理、可验证任务、格式约束、agent tool use，以及“能写出 reward function”的场景。

#### 3.1 GRPO算法的原理

和早期的 PPO 相比，GRPO 最大的优势是不需要 value model，因此显存占用更低，训练链路也更干净。这也是它在 reasoning / R1-style 训练中迅速流行的原因。

GRPO算法的训练流程一般为：

1. 对每个 prompt 采样 `G` 个 completion。
2. 用 reward function 或 reward model 分别打分，得到 `r_1, r_2, ..., r_G`。
3. 在组内做相对归一化，得到 advantage。
4. 用 clipped policy objective 更新 policy。
5. 如果 `beta > 0`，再加上相对 reference 的 KL 项。

组内 advantage 的直觉形式仍然是：

$$
\hat{A}_i = \frac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r)}
$$

这种方式规避了cirtic模型，使用组内相对优势进行计算，相比PPO更为简单。

#### 3.2 TRL对GRPO算法的支持

GRPO 的训练集必须包含 `prompt` 列。样本既可以是标准字符串，也可以是对话消息列表。额外字段不会参与模型输入，但会作为关键字参数传给 reward function。

标准格式示例：

```json
{
  "prompt": "解方程 2x + 3 = 7，并把推理写在 <think></think> 中，最终答案写在 <answer></answer> 中。",
  "ground_truth": "2"
}
```

对话格式示例：

```json
{
  "prompt": [
    {"role": "user", "content": "解方程 2x + 3 = 7，并把推理写在 <think></think> 中，最终答案写在 <answer></answer> 中。"}
  ],
  "ground_truth": "2"
}
```

这里的 `ground_truth` 不是保留字段，但只要它存在于数据集中，TRL 就会把它传给你的奖励函数。

TRL库为GRPO算法的实现提供了多种变体的支持：

1. `beta=0.0` 是默认值。也就是说，默认情况下**不会加载 reference model**，也不会启用 KL 惩罚。
2. `loss_type="dapo"` 是默认值，而不是最早论文里的原始 `grpo`。原因是原始 GRPO 有明显的长度偏置。
3. `scale_rewards="group"` 是默认值；如果你担心题目难度偏置，可以考虑 `False` 或 `"none"`。
4. `num_generations=8` 仍是常见默认组大小，但有效 batch size 必须能被它整除。


在`examples/scripts/grpo.py`下，TRL提供了GRPO训练的示例代码。

In [ ]:
import re
from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from trl.rewards import think_format_reward


def build_grpo_dataset():
    return Dataset.from_list(
        [
            {
                "prompt": [
                    {
                        "role": "user",
                        "content": "解方程 2x + 3 = 7。请把推理写在 <think></think> 中，最终答案写在 <answer></answer> 中。",
                    }
                ],
                "ground_truth": "2",
            },
            {
                "prompt": [
                    {
                        "role": "user",
                        "content": "解方程 3x - 5 = 10。请把推理写在 <think></think> 中，最终答案写在 <answer></answer> 中。",
                    }
                ],
                "ground_truth": "5",
            },
        ]
    )


def boxed_accuracy_reward(completions, ground_truth, **kwargs):
    rewards = []
    for completion, answer in zip(completions, ground_truth):
        if isinstance(completion, list):
            text = completion[0]["content"]
        else:
            text = completion

        match = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL)
        prediction = match.group(1).strip() if match else ""
        rewards.append(1.0 if prediction == answer else 0.0)
    return rewards


def run_grpo_example():
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    train_dataset = build_grpo_dataset()

    training_args = GRPOConfig(
        output_dir="./grpo_results",
        learning_rate=1e-6,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_generations=8,
        max_completion_length=256,
        beta=0.0,
        loss_type="dapo",
        scale_rewards="group",
        log_completions=True,
    )

    trainer = GRPOTrainer(
        model=model_name,
        reward_funcs=[think_format_reward, boxed_accuracy_reward],
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )
    return trainer


trainer = run_grpo_example()
trainer.train()